In [1]:
# import torch
import datajoint as dj
from experiments.dj.dataloader_tables import DataLoaderConfig, AltDataLoaderConfig
from experiments.dj.schema import schema
from experiments.dj.likelihood_tables import LikelihoodConfig

from experiments.dj.prior_tables import FlowPriorConfig, AdaptPriorConfig
from experiments.dj.sysident_tables import SIConfig
from experiments.dj.posterior_tables import SBVGPConfig, VPostPriorConfig
from experiments.dj.result_tables import (
    FlowPriorResult,
    FPSamples,
    FPSamplesConfig,
    LikelihoodResult,
    MLPCondSamples,
    MLPCondSamplesConfig,
    SBVGPResult2,
    SIResult,
    AdaptPriorResult,
    AdaptPriorSamplesConfig,
    SBVGPAdaptedResult,
    VPostPriorResult
)
from experiments.dj.evaluation_tables import SIEval, SBVGPEval
from experiments.dj.trainer_tables import (
    FPTrainerConfig,
    LLTrainerConfig,
    SBVGPTrainerConfig,
    SITrainerConfig,
    AdaptPriorTrainer,
    VPTrainerConfig
)

from experiments.dj.dj_helpers import drop_schema_dot_jobs
from experiments.dj.dj_helpers import fetch_best_model_results
from task_transfer.ml_lib.data_loading import build_dataloaders

from collections import OrderedDict

from task_transfer.utils.utils import dict_product
from experiments.dj.evaluation_tables import FlowPriorEval

[2025-04-07 10:29:17,886][INFO]: Connecting sshrinivasan@134.76.19.44:3306
[2025-04-07 10:29:17,902][INFO]: Connected sshrinivasan@134.76.19.44:3306


In [4]:
! dot -V

/bin/bash: dot: command not found


In [3]:
dj.Diagram(schema)

FileNotFoundError: [Errno 2] "dot" not found in path.

In [2]:
VPostPriorResult()

vp_id,trainer_id,dl_id,"train_var_marginal_mean mean per trial, per sample, in nats",train_var_marginal_sem standard error of the mean,val_var_marginal_mean,val_var_marginal_sem,test_var_marginal_mean,test_var_marginal_sem,"train_marginal_obs_ll_mean mean per trial, per sample, in nats",train_marginal_obs_ll_sem standard error of the mean,val_marginal_obs_ll_mean,val_marginal_obs_ll_sem,test_marginal_obs_ll_mean,test_marginal_obs_ll_sem,"train_post_ll_mean mean per trial, per sample, in nats",train_post_ll_sem standard error of the mean,val_post_ll_mean,val_post_ll_sem,test_post_ll_mean,test_post_ll_sem,"train_prior_ll_mean mean per trial, per sample, in nats",train_prior_ll_sem standard error of the mean,val_prior_ll_mean,val_prior_ll_sem,test_prior_ll_mean,test_prior_ll_sem,tracker_output,eval_output,model trained variational model (with joint and posterior)
01e9d80e496d22d5fc4b1d5e4c0efdb3,06ee7c0da04334f4dbad37a27d7d2fb5,3d740ef65d4ec3d651cb862eb90143df,-23.161558151245117,26.298738479614258,-33.45803451538086,31.834735870361328,-29.387657165527344,44.24879837036133,-48.08610153198242,88.90666961669922,-27.662734985351562,115.50831604003906,-12.151156425476074,100.80838775634766,-4.306342124938965,0.24633082747459412,-4.268877983093262,0.43999776244163513,-4.327520847320557,0.6242481470108032,-4.701415061950684,0.3025374114513397,-4.6410040855407715,0.5342189073562622,-4.695712566375732,0.7397690415382385,=BLOB=,=BLOB=,=BLOB=
01e9d80e496d22d5fc4b1d5e4c0efdb3,0cd3274d19c95ccff2db63c5644bae01,3d740ef65d4ec3d651cb862eb90143df,1352.235595703125,408.5346374511719,1340.5009765625,663.7172241210938,1253.314697265625,756.643798828125,-656.0164184570312,327.9098205566406,-633.1060180664062,528.89453125,-540.550048828125,520.9268798828125,-53.42240905761719,8.201157569885254,-52.5797119140625,14.551803588867188,-51.982872009277344,19.674169540405273,-31.209745407104492,5.265025615692139,-31.183094024658203,9.691041946411133,-29.542335510253906,11.911727905273438,=BLOB=,=BLOB=,=BLOB=
01e9d80e496d22d5fc4b1d5e4c0efdb3,13411240c1b15f4544e9bfb34462e23d,3d740ef65d4ec3d651cb862eb90143df,-32.15298843383789,20.36845588684082,-36.421875,34.805179595947266,-39.87627410888672,37.41229248046875,-46.269649505615234,95.78277587890625,-26.163103103637695,126.25935363769531,-19.267187118530273,133.15249633789062,-4.231546401977539,0.23553714156150818,-4.2003560066223145,0.42334312200546265,-4.263096332550049,0.6158022284507751,-4.653509140014648,0.2889823913574219,-4.606187343597412,0.5146780610084534,-4.660199165344238,0.7285693883895874,=BLOB=,=BLOB=,=BLOB=
01e9d80e496d22d5fc4b1d5e4c0efdb3,17a758ffc1c9f0f39c583000a790d8aa,3d740ef65d4ec3d651cb862eb90143df,264.6831359863281,134.10569763183594,243.10833740234375,213.05836486816406,233.1746063232422,255.23269653320312,-371.6274108886719,163.5964813232422,-332.320068359375,239.0276336669922,-316.91815185546875,280.4833679199219,-56.84566879272461,30.351530075073242,-49.28150939941406,18.309856414794922,-52.559654235839844,32.22270965576172,-24.58317756652832,3.3104333877563477,-24.81254005432129,6.355627536773682,-23.47262954711914,7.866446018218994,=BLOB=,=BLOB=,=BLOB=
01e9d80e496d22d5fc4b1d5e4c0efdb3,1901b76d2f4073ddab068fdd17c89ae6,3d740ef65d4ec3d651cb862eb90143df,-24.38449478149414,28.77837371826172,-29.789623260498047,32.37923049926758,-31.095977783203125,41.04191589355469,-45.355831146240234,86.92366790771484,-27.713848114013672,145.3661346435547,-17.9427433013916,110.937744140625,-4.277206897735596,0.2497919201850891,-4.2428507804870605,0.44808197021484375,-4.307188034057617,0.6351763010025024,-4.6714911460876465,0.29780977964401245,-4.618204116821289,0.5248755216598511,-4.673855781555176,0.7337208390235901,=BLOB=,=BLOB=,=BLOB=
01e9d80e496d22d5fc4b1d5e4c0efdb3,1bb5794d17f54f9c5de36c437b92c679,3d740ef65d4ec3d651cb862eb90143df,-24.819473266601562,22.7443790435791,-28.64702796936035,41.27439498901367,-32.92975616455078,38.95169448852539,-54.250267028808594,103.88996124267578,-44.803375244140625,156

In [3]:
AltDataLoaderConfig()

id,data_fname,train_prop,val_prop
260a5ea8175f75eaef132f42873ad14a,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_1_dataset.pkl,0.7,0.2
3d740ef65d4ec3d651cb862eb90143df,/src/project/data/synthetic/haefner_2afc/haefner_model_4neuron_task2_dataset.pkl,0.7,0.2
5352c4a57ef18797b082283de593157b,/src/project/data/synthetic/haefner_2afc/haefner_model_4neuron_highdelta_task2_dataset.pkl,0.7,0.2
592885da0624c8a8c3073ec47d9bcfba,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_task2_dataset.pkl,0.7,0.2
94efb58694007205fac996d7963f88c5,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task2_dataset.pkl,0.7,0.2
9ef3ae6fea33eba634d928a88b866836,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_2_dataset.pkl,0.7,0.2
b8379e7d6998fc94a08a9a3742eec12d,/src/project/data/synthetic/haefner_2afc/flat_haefner_dataset.pkl,0.7,0.2
d6b36dc9d4024882e4b7ccc597495a32,/src/project/data/synthetic/haefner_2afc/flat_haefner_100k_dataset.pkl,0.7,0.2
d74090584b0b974c4444a5ec64c3d87d,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_highdelta_task2_dataset.pkl,0.7,0.2
e248f1c663817cdabf379015476a3a2e,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_task2_dataset.pkl,0.7,0.2


In [7]:
DataLoaderConfig()

id,data_fname,train_prop,val_prop
05977a317062b759857ee411a2e60648,/src/project/data/synthetic/haefner_2afc/haefner_2neuron_task1.pkl,0.7,0.2
260a5ea8175f75eaef132f42873ad14a,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_1_dataset.pkl,0.7,0.2
4477b5e82704db0bc19727864c7ef5aa,/src/project/data/synthetic/haefner_2afc/haefner_4neuron_task1.pkl,0.7,0.2
5352c4a57ef18797b082283de593157b,/src/project/data/synthetic/haefner_2afc/haefner_model_4neuron_highdelta_task2_dataset.pkl,0.7,0.2
8e9be142eedb21007255e89dbff362da,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_highdelta_task1_dataset.pkl,0.7,0.2
94efb58694007205fac996d7963f88c5,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task2_dataset.pkl,0.7,0.2
9ef3ae6fea33eba634d928a88b866836,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_2_dataset.pkl,0.7,0.2
b8379e7d6998fc94a08a9a3742eec12d,/src/project/data/synthetic/haefner_2afc/flat_haefner_dataset.pkl,0.7,0.2
bb9bdd1ccd59e5a8c801d7f2d43e0317,/src/project/data/synthetic/haefner_2afc/haefner_model_4neuron_highdelta_task1_dataset.pkl,0.7,0.2
d74090584b0b974c4444a5ec64c3d87d,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_highdelta_task2_dataset.pkl,0.7,0.2
